# 计算交叉谱数据 - 降水与不同气压层散度

本笔记本用于计算降水与不同气压层散度的交叉谱数据，并保存到 `fig11/` 目录

**注意**: 此脚本只需要运行一次来生成数据文件，之后使用 `fig11_plot_crossspectrum_clean.ipynb` 进行绘图

In [1]:
import xarray as xr
import numpy as np
import sys
import os
from pathlib import Path

# 导入wave_tools
WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))
from wave_tools import calculate_cross_spectrum, remove_annual_cycle

print("✅ 导入模块成功")

✅ 导入模块成功


In [2]:
# ============ 配置 ============

# 数据路径
DATA_DIR = '/work/mh1498/m301257/data_origin'
DIV_DIR = '/work/mh1498/m301257/3D_data/divergence_2d_plvls'
MASK_PATH = '../processed_data/land_mask_2deg.nc'

# 输出路径
OUTPUT_DIR = 'fig11'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 要分析的气压层
PRESSURE_LEVELS = [250, 500, 850]  # hPa

# 实验名称
EXPERIMENTS = ['cntl', 'p4k', '4co2']

print(f"输出目录: {OUTPUT_DIR}")
print(f"气压层: {PRESSURE_LEVELS} hPa")
print(f"实验: {EXPERIMENTS}")

输出目录: fig11
气压层: [250, 500, 850] hPa
实验: ['cntl', 'p4k', '4co2']


In [3]:
# ============ 加载海洋掩膜 ============

fraction = xr.open_dataarray(MASK_PATH)
ocean_mask = (fraction == 0)

print(f"✅ 海洋掩膜加载成功")
print(f"   形状: {ocean_mask.shape}")

✅ 海洋掩膜加载成功
   形状: (15, 180)


In [4]:
# ============ 加载数据 ============

# 加载降水数据
pr_data = {}
for exp in EXPERIMENTS:
    file_path = os.path.join(DATA_DIR, f'pr_{exp}_2deg_interp.nc')
    print(f"加载 {exp.upper()} 降水数据...")
    pr = xr.open_dataarray(file_path, chunks={'time': 100})
    pr_data[exp] = pr.where(ocean_mask, drop=True)
    print(f"  ✓ {pr_data[exp].shape}")

# 加载散度数据
div_data = {}
for exp in EXPERIMENTS:
    file_path = os.path.join(DIV_DIR, f'div_pressure_levels_{exp}.nc')
    print(f"加载 {exp.upper()} 散度数据...")
    div_data[exp] = xr.open_dataset(file_path)
    print(f"  ✓ {list(div_data[exp].dims.keys())}")

print("\n✅ 所有数据加载完成")

加载 CNTL 降水数据...
  ✓ (5114, 15, 167)
加载 P4K 降水数据...
  ✓ (5114, 15, 167)
加载 4CO2 降水数据...
  ✓ (5114, 15, 167)
加载 CNTL 散度数据...
  ✓ ['plev', 'time', 'lat', 'lon']
加载 P4K 散度数据...
  ✓ (5114, 15, 167)
加载 CNTL 散度数据...
  ✓ ['plev', 'time', 'lat', 'lon']
加载 P4K 散度数据...


/tmp/ipykernel_1548429/371342308.py:18: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  ✓ {list(div_data[exp].dims.keys())}")
/tmp/ipykernel_1548429/371342308.py:18: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  ✓ {list(div_data[exp].dims.keys())}")


  ✓ ['plev', 'time', 'lat', 'lon']
加载 4CO2 散度数据...
  ✓ ['plev', 'time', 'lat', 'lon']

✅ 所有数据加载完成
  ✓ ['plev', 'time', 'lat', 'lon']

✅ 所有数据加载完成


/tmp/ipykernel_1548429/371342308.py:18: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  ✓ {list(div_data[exp].dims.keys())}")


In [5]:
# ============ 计算交叉谱 ============

crossspectrum_results = {}

for exp_name in EXPERIMENTS:
    print(f"\n{'='*60}")
    print(f"处理 {exp_name.upper()}...")
    print(f"{'='*60}")
    
    pr_raw = pr_data[exp_name]
    div_dataset = div_data[exp_name]
    
    crossspectrum_results[exp_name] = {}
    
    for plev in PRESSURE_LEVELS:
        print(f"\n  --- 气压层: {plev} hPa ---")
        
        # 选择气压层
        div_raw = div_dataset['__xarray_dataarray_variable__'].sel(plev=plev)
        
        # 网格对齐
        if len(pr_raw.lon) != len(div_raw.lon):
            print(f"    对齐经度网格...")
            div_raw = div_raw.interp(lon=pr_raw.lon, method='linear')
        
        if len(pr_raw.lat) != len(div_raw.lat):
            print(f"    对齐纬度网格...")
            div_raw = div_raw.interp(lat=pr_raw.lat, method='linear')
        
        # 应用海洋掩膜
        pr_masked = pr_raw.where(ocean_mask).fillna(0)
        div_masked = div_raw.where(ocean_mask).fillna(0)
        
        # 去除年循环
        print(f"    去除年循环...")
        pr_ano = remove_annual_cycle(pr_masked)
        div_ano = remove_annual_cycle(div_masked)
        
        # 计算交叉谱
        print(f"    计算交叉谱...")
        result = calculate_cross_spectrum(
            pr_ano, div_ano,
            segLen=96,
            segOverLap=-65,
            symmetry='symm',
            return_xarray=True
        )
        
        if result is None:
            print(f"    ✗ 失败")
            continue
        
        # 存储结果
        crossspectrum_results[exp_name][plev] = {
            'STC': result['STC'],
            'freq': result['freq'],
            'wave': result['wave'],
            'nseg': result['nseg'],
            'dof': result['dof'],
            'p': result['p'],
            'prob_coh2': result['prob_coh2']
        }
        
        print(f"    ✓ 完成 - STC shape: {result['STC'].shape}")

print("\n" + "="*60)
print("✅ 所有交叉谱计算完成！")
print("="*60)


处理 CNTL...

  --- 气压层: 250 hPa ---
    对齐经度网格...
    对齐纬度网格...
    对齐纬度网格...
    去除年循环...
    去除年循环...
    计算交叉谱...
    计算交叉谱...
    ✓ 完成 - STC shape: (8, 49, 167)

  --- 气压层: 500 hPa ---
    对齐经度网格...
    ✓ 完成 - STC shape: (8, 49, 167)

  --- 气压层: 500 hPa ---
    对齐经度网格...
    对齐纬度网格...
    对齐纬度网格...
    去除年循环...
    去除年循环...
    计算交叉谱...
    计算交叉谱...
    ✓ 完成 - STC shape: (8, 49, 167)

  --- 气压层: 850 hPa ---
    对齐经度网格...
    ✓ 完成 - STC shape: (8, 49, 167)

  --- 气压层: 850 hPa ---
    对齐经度网格...
    对齐纬度网格...
    对齐纬度网格...
    去除年循环...
    去除年循环...
    计算交叉谱...
    计算交叉谱...
    ✓ 完成 - STC shape: (8, 49, 167)

处理 P4K...

  --- 气压层: 250 hPa ---
    对齐经度网格...
    ✓ 完成 - STC shape: (8, 49, 167)

处理 P4K...

  --- 气压层: 250 hPa ---
    对齐经度网格...
    对齐纬度网格...
    对齐纬度网格...
    去除年循环...
    去除年循环...
    计算交叉谱...
    计算交叉谱...
    ✓ 完成 - STC shape: (8, 49, 167)

  --- 气压层: 500 hPa ---
    对齐经度网格...
    ✓ 完成 - STC shape: (8, 49, 167)

  --- 气压层: 500 hPa ---
    对齐经度网格...
    对齐纬度网格...
    对齐纬度网格

In [6]:
# ============ 保存数据 ============

output_path = os.path.join(OUTPUT_DIR, 'crossspectrum_pr_div_pressure_levels.npz')

# 准备保存的数据
save_dict = {}
for exp_name in EXPERIMENTS:
    for plev in PRESSURE_LEVELS:
        if plev in crossspectrum_results[exp_name]:
            key = f"{exp_name}_{plev}"
            save_dict[key] = crossspectrum_results[exp_name][plev]

np.savez(output_path, **save_dict)

print(f"\n✅ 数据已保存到: {output_path}")
print(f"   文件大小: {os.path.getsize(output_path) / 1024**2:.2f} MB")
print(f"\n包含的数据:")
for key in save_dict.keys():
    print(f"  - {key}")


✅ 数据已保存到: fig11/crossspectrum_pr_div_pressure_levels.npz
   文件大小: 4.54 MB

包含的数据:
  - cntl_250
  - cntl_500
  - cntl_850
  - p4k_250
  - p4k_500
  - p4k_850
  - 4co2_250
  - 4co2_500
  - 4co2_850
